In [1]:
import numpy as np
from scipy.optimize import minimize, milp, LinearConstraint, Bounds

# Standard E12 & E24 Reihen (JLCPCB Basic C0G/NP0 Kondensatoren & Wirewound Induktivitäten)
E12 = np.array([1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2])
E24 = np.array([1.0, 1.1, 1.2, 1.3, 1.5, 1.6, 1.8, 2.0, 2.2, 2.4, 2.7, 3.0,
                3.3, 3.6, 3.9, 4.3, 4.7, 5.1, 5.6, 6.2, 6.8, 7.5, 8.2, 9.1])

CAP_POOL = np.concatenate([E24 * d for d in [1e-12, 10e-12, 100e-12, 1e-9]])
CAP_POOL = CAP_POOL[(CAP_POOL >= 4.7e-12) & (CAP_POOL <= 3300e-12)]

IND_POOL = np.concatenate([E12 * d for d in [10e-9, 100e-9, 1e-6]])
IND_POOL = IND_POOL[(IND_POOL >= 47e-9) & (IND_POOL <= 2200e-9)]

# --- ABCD-Zweitor-Bausteine ---
def abcd_shunt_c(C, w, Q_c=1000.0):
    Y = 1j * w * C + (w * C / Q_c)
    return np.array([[1.0, 0.0], [Y, 1.0]], dtype=complex)

def abcd_series_l(L, w, Q_l=60.0):
    Z = (w * L / Q_l) + 1j * w * L
    return np.array([[1.0, Z], [0.0, 1.0]], dtype=complex)

def abcd_notch_lc(L, C, w, Q_l=60.0):
    Z_l = (w * L / Q_l) + 1j * w * L
    Z = 1.0 / (1.0 / Z_l + 1j * w * C)
    return np.array([[1.0, Z], [0.0, 1.0]], dtype=complex)

def eval_abcd(M, Zs=50.0, Zl=50.0):
    A, B, C, D = M[0, 0], M[0, 1], M[1, 0], M[1, 1]
    denom = A * Zl + B + C * Zs * Zl + D * Zs
    s21 = 10.0 * np.log10(4.0 * Zs * Zl / (np.abs(denom)**2) + 1e-18)
    Zin = (A * Zl + B) / (C * Zl + D)
    s11 = 20.0 * np.log10(np.abs((Zin - Zs) / (Zin + Zs)) + 1e-18)
    return s21, s11

# --- Topologie-Definitionen ---
def cascade_topology(topo, x, w, Q_l):
    if topo == "LMatch_WB2CBA":
        # L-Match (C_m, L_m) + WB2CBA Pi-LPF (C1, L2, C3, L4, C5)
        cm, lm, c1, l2, c3, l4, c5 = x
        return (abcd_shunt_c(cm, w) @ abcd_series_l(lm, w, Q_l) @
                abcd_shunt_c(c1, w) @ abcd_series_l(l2, w, Q_l) @
                abcd_shunt_c(c3, w) @ abcd_series_l(l4, w, Q_l) @
                abcd_shunt_c(c5, w))
    elif topo == "JA2GQP":
        # 7-Element Notch LPF: C1 -> (L1||C2) -> C3 -> (L2||C4) -> C5
        c1, l1, c2, c3, l2, c4, c5 = x
        return (abcd_shunt_c(c1, w) @ abcd_notch_lc(l1, c2, w, Q_l) @
                abcd_shunt_c(c3, w) @ abcd_notch_lc(l2, c4, w, Q_l) @
                abcd_shunt_c(c5, w))
    elif topo == "4Element":
        # 4-Element LPF: C1 -> L2 -> C3 -> L4
        c1, l2, c3, l4 = x
        return (abcd_shunt_c(c1, w) @ abcd_series_l(l2, w, Q_l) @
                abcd_shunt_c(c3, w) @ abcd_series_l(l4, w, Q_l))
    raise ValueError(f"Unbekannte Topologie: {topo}")

TOPO_SPECS = {
    "LMatch_WB2CBA": {
        "names": ["Cm", "Lm", "C1", "L2", "C3", "L4", "C5"],
        "is_ind": [False, True, False, True, False, True, False],
        "seed": [180e-12, 220e-9, 220e-12, 680e-9, 470e-12, 680e-9, 220e-12],
    },
    "JA2GQP": {
        "names": ["C1", "L1", "C2", "C3", "L2", "C4", "C5"],
        "is_ind": [False, True, False, False, True, False, False],
        "seed": [390e-12, 120e-9, 250e-12, 820e-12, 330e-9, 43e-12, 390e-12],
    },
    "4Element": {
        "names": ["C1", "L2", "C3", "L4"],
        "is_ind": [False, True, False, True],
        "seed": [220e-12, 820e-9, 390e-12, 470e-9],
    },
}

# --- Monte-Carlo-Verifikation (±5 % auf alle L & C) ---
def run_mc(topo, x_nom, f0, Zs, Zl, harm_dbc, Q_l=60.0, fcc_limit=-43.0, tol=0.05, n_runs=1000):
    rng = np.random.default_rng(42)
    w0 = 2.0 * np.pi * f0
    w_harms = [k * w0 for k in range(2, 2 + len(harm_dbc))]
    wc_s11, wc_s21, wc_m = -999.0, 0.0, 999.0

    for _ in range(n_runs):
        pert = [v * rng.uniform(1.0 - tol, 1.0 + tol) for v in x_nom]
        s21, s11 = eval_abcd(cascade_topology(topo, pert, w0, Q_l), Zs, Zl)
        wc_s11 = max(wc_s11, s11)
        wc_s21 = min(wc_s21, s21)
        for wh, hl in zip(w_harms, harm_dbc):
            s21_h, _ = eval_abcd(cascade_topology(topo, pert, wh, Q_l), Zs, Zl)
            wc_m = min(wc_m, fcc_limit - (hl + s21_h))
    return wc_s11, wc_s21, wc_m

# --- Two-Stage NLP + MILP Solver ---
def solve_topology_mip(topo, f0, Zs=50.0, Zl=50.0, harm_dbc=(-17.6, -31.1, -43.0, -46.0),
                       Q_l=60.0, fcc_limit=-43.0, max_caps_per_node=2, reel_penalty=0.25):
    spec = TOPO_SPECS[topo]
    names, is_ind = spec["names"], spec["is_ind"]
    scale = (14.1e6 / f0)
    x_init = np.array(spec["seed"]) * scale

    w0 = 2.0 * np.pi * f0
    w2, w3 = 2.0 * w0, 3.0 * w0
    w_harms = [k * w0 for k in range(2, 2 + len(harm_dbc))]

    def eval_net(x):
        s21_0, s11_0 = eval_abcd(cascade_topology(topo, x, w0, Q_l), Zs, Zl)
        margins = [fcc_limit - (hl + eval_abcd(cascade_topology(topo, x, wh, Q_l), Zs, Zl)[0])
                   for wh, hl in zip(w_harms, harm_dbc)]
        return s21_0, s11_0, min(margins)

    # 1. NLP-Seed mit Passband-Stabilitätsmargin (f_pass = 1.12 * f0 mitprüfen!)
    w_hi = 1.12 * w0
    def obj_nlp(u):
        x = np.exp(u)
        s21, s11, min_m = eval_net(x)
        s21_hi, s11_hi = eval_abcd(cascade_topology(topo, x, w_hi, Q_l), Zs, Zl)
        # S11 nicht unter -26 dB drücken, dafür S11 bei 1.12*f0 < -20 dB halten!
        pen = (max(0.0, 14.0 - min_m)**2 * 50.0 +
               max(0.0, -0.42 - s21)**2 * 200.0 +
               max(0.0, s11_hi + 20.0)**2 * 10.0)
        if topo == "JA2GQP":
            f_n1 = 1.0 / (2.0 * np.pi * np.sqrt(x[1] * x[2]))
            f_n2 = 1.0 / (2.0 * np.pi * np.sqrt(x[4] * x[5]))
            pen += ((f_n1 - 2 * f0) / f0)**2 * 500.0 + ((f_n2 - 3 * f0) / f0)**2 * 500.0
        return max(s11, -28.0) + pen

    res_nlp = minimize(obj_nlp, np.log(x_init), method="Nelder-Mead",
                       options={"maxiter": 4000, "xatol": 1e-5, "fatol": 1e-4})
    x_star = np.exp(res_nlp.x)
    s21_real, s11_real, m_real = eval_net(x_star)

    # 2. MILP Diskretisierung auf JLCPCB Basic E24 (bis zu 2 parallel) & E12 Spulen
    Nc, Nl = len(CAP_POOL), len(IND_POOL)
    offsets = []
    cur = 0
    for ind in is_ind:
        offsets.append(cur)
        cur += Nl if ind else Nc
    idx_u = cur
    n_vars = cur + Nl  # Binärvariablen u_reel[k] für jede einzigartige Spulenrolle

    c_obj = np.zeros(n_vars)
    for i, (ind, target) in enumerate(zip(is_ind, x_star)):
        pool = IND_POOL if ind else CAP_POOL
        c_obj[offsets[i]:offsets[i] + len(pool)] = np.abs(pool - target) / target
    c_obj[idx_u:idx_u + Nl] = reel_penalty  # $3.00 Feeder-Gebühr pro Spulenrolle bestrafen

    rows, lb, ub = [], [], []
    # Auswahl-Constraints pro Bauteilposition
    for i, ind in enumerate(is_ind):
        pool = IND_POOL if ind else CAP_POOL
        r = np.zeros(n_vars)
        r[offsets[i]:offsets[i] + len(pool)] = 1.0
        rows.append(r)
        lb.append(1.0)
        ub.append(1.0 if ind else float(max_caps_per_node))

    # Spulenrollen-Kopplung: b_l[i, k] <= u_reel[k]
    for i, ind in enumerate(is_ind):
        if ind:
            for k in range(Nl):
                r = np.zeros(n_vars)
                r[offsets[i] + k] = 1.0
                r[idx_u + k] = -1.0
                rows.append(r); lb.append(-np.inf); ub.append(0.0)

    # Toleranzfenster um Real-Opt (±8 %, bei JA2GQP-Notch-Cs ±4 % für scharfe Kerbe)
    for i, (ind, target) in enumerate(zip(is_ind, x_star)):
        pool = IND_POOL if ind else CAP_POOL
        tol_win = 0.04 if (topo == "JA2GQP" and i in (2, 5)) else 0.08
        r = np.zeros(n_vars)
        r[offsets[i]:offsets[i] + len(pool)] = pool
        rows.append(r); lb.append((1.0 - tol_win) * target); ub.append((1.0 + tol_win) * target)

    sol = milp(c=c_obj, integrality=np.ones(n_vars),
               bounds=Bounds(np.zeros(n_vars), np.ones(n_vars)),
               constraints=LinearConstraint(np.array(rows), lb, ub))

    if not sol.success:
        print(f"  {topo:<15}: MILP kein Treffer im ±8%-Fenster (Real Opt: S11={s11_real:.2f} dB, S21={s21_real:.3f} dB)")
        return

    v = np.round(sol.x).astype(int)
    x_disc, part_strs = [], []
    for i, (ind, nm) in enumerate(zip(is_ind, names)):
        pool = IND_POOL if ind else CAP_POOL
        sel = v[offsets[i]:offsets[i] + len(pool)]
        val = float(np.dot(sel, pool))
        x_disc.append(val)
        picks = [pool[k] for k in np.where(sel > 0)[0]]
        lbl = "+".join(f"{p*1e9:.0f}n" if ind else f"{p*1e12:.1f}p" for p in picks)
        part_strs.append(f"{nm}={lbl}")

    n_reels = int(np.sum(v[idx_u:idx_u + Nl]))
    s21_d, s11_d, m_d = eval_net(x_disc)
    wc_s11, wc_s21, wc_m = run_mc(topo, x_disc, f0, Zs, Zl, harm_dbc, Q_l=Q_l, fcc_limit=fcc_limit)

    print(f"  {topo:<15} (Real Opt: S11={s11_real:6.2f} dB, S21={s21_real:6.3f} dB)")
    print(f"    -> MILP Basic  : S11={s11_d:6.2f} dB, S21={s21_d:6.3f} dB, FCC=+{m_d:4.1f} dB | "
          f"Spulen-Rollen={n_reels} (${3*n_reels:.2f})")
    print(f"    -> ±5% MC (WC) : S11={wc_s11:6.2f} dB, S21={wc_s21:6.3f} dB, FCC=+{wc_m:4.1f} dB")
    print(f"    -> Bauteile    : {', '.join(part_strs)}\n")

for band_name, f0 in [("20m (14.10 MHz)", 14.1e6), ("40m (7.05 MHz)", 7.05e6)]:
    print(f"==========================================================================")
    print(f" BAND: {band_name} | Zs=50.0Ω -> Zl=50.0Ω | Q_L=60 | E24 Basic Caps")
    print(f"==========================================================================")
    for topo in ["LMatch_WB2CBA", "JA2GQP", "4Element"]:
        solve_topology_mip(topo, f0, Zs=50.0, Zl=50.0)


 BAND: 20m (14.10 MHz) | Zs=50.0Ω -> Zl=50.0Ω | Q_L=60 | E24 Basic Caps
  LMatch_WB2CBA  : MILP kein Treffer im ±8%-Fenster (Real Opt: S11=-28.00 dB, S21=-0.540 dB)
  JA2GQP          (Real Opt: S11=-28.61 dB, S21=-0.285 dB)
    -> MILP Basic  : S11=-22.48 dB, S21=-0.314 dB, FCC=+31.4 dB | Spulen-Rollen=1 ($3.00)
    -> ±5% MC (WC) : S11=-15.99 dB, S21=-0.436 dB, FCC=+15.0 dB
    -> Bauteile    : C1=200.0p, L1=390n, C2=82.0p, C3=360.0p, L2=390n, C4=33.0p, C5=150.0p

  4Element        (Real Opt: S11=-12.03 dB, S21=-1.301 dB)
    -> MILP Basic  : S11= -4.05 dB, S21=-3.173 dB, FCC=+11.8 dB | Spulen-Rollen=2 ($6.00)
    -> ±5% MC (WC) : S11= -2.11 dB, S21=-5.071 dB, FCC=+10.0 dB
    -> Bauteile    : C1=620.0p, L2=680n, C3=270.0p, L4=1800n

 BAND: 40m (7.05 MHz) | Zs=50.0Ω -> Zl=50.0Ω | Q_L=60 | E24 Basic Caps
  LMatch_WB2CBA  : MILP kein Treffer im ±8%-Fenster (Real Opt: S11=-28.01 dB, S21=-0.539 dB)
  JA2GQP          (Real Opt: S11=-28.00 dB, S21=-0.359 dB)
    -> MILP Basic  : S11=-16.08 